In [1]:
import pandas as pd
import os
import requests
from tqdm import tqdm
import json

In [2]:
df = pd.read_csv(r"D:\Data Science\speech&audio\FT Data - data.csv")

In [5]:
import os

# Go one level up (to main SPEECH&AUDIO folder)
base_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

# Create subdirectories
os.makedirs(os.path.join(base_dir, "audio"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "transcripts_raw"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "metadata"), exist_ok=True)

print("✅ Folder structure created successfully in main SPEECH&AUDIO folder!")


✅ Folder structure created successfully in main SPEECH&AUDIO folder!


In [6]:
import os
import requests
import pandas as pd
from tqdm import tqdm

# === Step 1: Set base directory ===
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
AUDIO_DIR = os.path.join(BASE_DIR, "audio")
TRANS_DIR = os.path.join(BASE_DIR, "transcripts_raw")
META_DIR = os.path.join(BASE_DIR, "metadata")

# Ensure folders exist
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TRANS_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

print("✅ Folder structure ready at:", BASE_DIR)


# === Step 2: Define download function ===
def download_file(url, path):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
        return True
    except Exception as e:
        print(f"❌ Failed to download {url}: {e}")
        return False


# === Step 3: Load dataset ===
df = pd.read_csv(os.path.join(os.getcwd(), "..", "FT Data - data.csv"))
print(f"📊 Loaded dataset with {len(df)} rows")


# === Step 4: Download files ===
for i, row in tqdm(df.iterrows(), total=len(df), desc="Downloading data"):
    rec_id = str(row["recording_id"])

    # Audio file
    if pd.notna(row.get("rec_url_gcp")):
        audio_path = os.path.join(AUDIO_DIR, f"{rec_id}.wav")
        download_file(row["rec_url_gcp"], audio_path)

    # Transcript JSON
    if pd.notna(row.get("transcription_url_gcp")):
        trans_path = os.path.join(TRANS_DIR, f"{rec_id}.json")
        download_file(row["transcription_url_gcp"], trans_path)

    # Metadata JSON
    if pd.notna(row.get("metadata_url_gcp")):
        meta_path = os.path.join(META_DIR, f"{rec_id}.json")
        download_file(row["metadata_url_gcp"], meta_path)

print("🎯 All available files downloaded successfully!")


✅ Folder structure ready at: d:\Data Science\speech&audio\data
📊 Loaded dataset with 104 rows


🎯 All available files downloaded successfully!


In [10]:
import os
import json
from tqdm import tqdm

# Move up one directory from the notebook to the project root
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Define correct absolute paths
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "transcripts_raw")
TXT_DIR = os.path.join(DATA_DIR, "transcripts_txt")

# Ensure folders exist
os.makedirs(TXT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("TXT_DIR:", TXT_DIR)

# Convert all transcript JSONs to TXT
for file in tqdm(os.listdir(RAW_DIR), desc="Converting transcripts"):
    if not file.endswith(".json"):
        continue

    raw_path = os.path.join(RAW_DIR, file)
    out_path = os.path.join(TXT_DIR, file.replace(".json", ".txt"))

    try:
        with open(raw_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            text = " ".join([seg.get("text", "").strip() for seg in data if seg.get("text")])
        elif isinstance(data, dict):
            text = data.get("text", "").strip()
        else:
            text = ""

        with open(out_path, "w", encoding="utf-8") as out:
            out.write(text)

    except Exception as e:
        print(f"[ERROR] Could not parse {file}: {e}")


BASE_DIR: d:\Data Science\speech&audio
RAW_DIR: d:\Data Science\speech&audio\data\transcripts_raw
TXT_DIR: d:\Data Science\speech&audio\data\transcripts_txt


Converting transcripts: 100%|██████████| 104/104 [00:00<00:00, 1238.75it/s]


In [12]:
import os
from tqdm import tqdm
from datasets import Dataset

# Go one level up from notebook folder
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

AUDIO_DIR = os.path.join(BASE_DIR, "data", "audio")
TXT_DIR = os.path.join(BASE_DIR, "data", "transcripts_txt")
OUT_PATH = os.path.join(BASE_DIR, "data", "dataset_ready")

# Debug check
print("Audio Dir:", AUDIO_DIR)
print("Txt Dir:", TXT_DIR)

# Verify existence before proceeding
if not os.path.exists(AUDIO_DIR):
    raise FileNotFoundError(f"❌ Audio directory not found: {AUDIO_DIR}")

if not os.path.exists(TXT_DIR):
    raise FileNotFoundError(f"❌ Transcript directory not found: {TXT_DIR}")

# Collect matching audio/text files
audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")])

data = {"audio": [], "sentence": []}

for file in tqdm(audio_files, desc="Preparing dataset"):
    audio_path = os.path.join(AUDIO_DIR, file)
    txt_path = os.path.join(TXT_DIR, file.replace(".wav", ".txt"))

    if not os.path.exists(txt_path):
        print(f"❌ Missing transcript for {file}")
        continue

    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        print(f"⚠️ Empty transcript for {file}")
        continue

    data["audio"].append(audio_path)
    data["sentence"].append(text)

print(f"✅ Collected {len(data['audio'])} valid pairs")

# Convert to Hugging Face Dataset
dataset = Dataset.from_dict(data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Save dataset
os.makedirs(OUT_PATH, exist_ok=True)
dataset.save_to_disk(OUT_PATH)

print(f"📦 Dataset saved at: {OUT_PATH}")


Audio Dir: d:\Data Science\speech&audio\data\audio
Txt Dir: d:\Data Science\speech&audio\data\transcripts_txt


Preparing dataset: 100%|██████████| 104/104 [00:00<00:00, 129.31it/s]


✅ Collected 104 valid pairs


Saving the dataset (1/1 shards): 100%|██████████| 11/11 [00:00<00:00, 1375.22 examples/s]

📦 Dataset saved at: d:\Data Science\speech&audio\data\dataset_ready
